# Conditional VAE modeling for single cell data

## Data loading

Configure root.

In [ ]:
import sys, subprocess, os
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
import importlib
import seaborn as sns
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/singlecell-autoencoder.git"
repo_dir = Path("singlecell-autoencoder")
if COLAB:
    root = Path("/content/singlecell-autoencoder")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
generator = torch.Generator().manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Configure single-cell data path.

In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
    sc_path = str(data_dir / "single_cell")

else:
    sc_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/single_cell"

Load scRNA-seq data as CPMs.

In [ ]:
from src.sc_data import get_sc_data, df_to_tensors

data = get_sc_data(path = sc_path)

## Train

Train-test split.

In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split
test_mask = (data["dose"] == 2) & (data["timepoint"] == 2)
train_df = data[~test_mask]
test_df = data[test_mask]

# Train-val split
train_df, val_df = train_test_split(
    train_df,
    test_size = 0.2,
    random_state = 111,
    shuffle = True
)

# Calculate proportions
num_data = data.shape[0]
train_prop = int(round(train_df.shape[0] * 100 / num_data, 0))
val_prop = int(round(val_df.shape[0] * 100 / num_data, 0))
test_prop = int(round(test_df.shape[0] * 100 / num_data, 0))

print(f"Train:val:test = {train_prop}:{val_prop}:{test_prop}")

Model + training params.

In [ ]:
# Get number of genes
num_genes = data.iloc[:, data.columns.str.contains("SP")].shape[1]

# Model params
model_params = {
    "input_dim": num_genes,
    "hidden_dim": 256,
    "latent_dim": 32
}

# Training params
batch_size = 16
epochs = 150
lr = 0.001
kl_weight = 0.1
adv_weight = 0.1
seed = 111

CVAE training loop.

In [ ]:
import src.train; importlib.reload(src.train)
from src.train import train_cvae

out = train_cvae(
    train_df = train_df, 
    val_df= val_df,
    batch_size = batch_size,
    epochs = epochs, 
    lr = lr, 
    model_params = model_params, 
    device = device,
    kl_weight = kl_weight,
    adv_weight = adv_weight,
    seed = seed    
)

model, train_vae_losses, train_adv_losses, val_vae_losses, val_adv_losses = out

Plot VAE loss over time.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_vae_losses, label = "Train")
plt.plot(val_vae_losses, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel("VAE loss")
plt.legend()

Plot adversarial loss over time.

In [ ]:
plt.plot(train_adv_losses, label = "Train")
plt.plot(val_adv_losses, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel("Adversarial loss")
plt.legend()

Save model from Colab.

In [ ]:
model.eval()
checkpoint_path = Path(
    "/content/drive/MyDrive/phenotype-prediction-data/CVAE_checkpoint.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_params": model_params,
    },
    checkpoint_path,
)
from google.colab import files
files.download("/content/CVAE_checkpoint.pt")

print(f"Saved to: {checkpoint_path}")
print(f"Exists: {checkpoint_path.exists()}")
print(f"Size: {checkpoint_path.stat().st_size:,} bytes")